# **Archetype Clustering & Shot Zone Aggregation Workflow**

---

### Data Pipeline Overview
This step merges preprocessed shot log data with player archetype assignments to calculate localized shooting efficiencies for each cluster.

* **Data Merging**: Joins the 2014-15 preprocessed shot logs with player cluster mappings (`ARCHETYPE_CLUSTER`) using `PLAYER_ID` as the relational key.
* **Output Artifact**: Exports the joined dataset to `data/shotlog_2014-15_with_clusters.csv` for downstream visualization scripts.

---

### Shot Zone Aggregation Logic
To profile the offensive footprint of each player archetype, the dataset is aggregated using a multi-level group-by operation:

* **Group Keys**: Segmented by `SHOT_ZONE_BASIC`, `SHOT_ZONE_AREA`, and `ARCHETYPE_CLUSTER`.
* **Metrics Calculated**:
  * `count`: Total field goal attempts (FGA) taken by the cluster in that specific zone.
  * `sum`: Total field goals made (FGM) by the cluster in that specific zone.
  * `ACCURACY`: Calculated absolutely as $\frac{\text{FGM}}{\text{FGA}}$ (`sum` / `count`).

**Final Export:** The resulting breakdown is saved directly to `cluster_zone_stat.csv` to feed the 2x3 subplot heatmap rendering function.


In [ ]:
import pandas as pd

data = pd.read_csv('data/shotlog_2014-15_preprocessed.csv')
player_cluster = pd.read_csv('data/player_archetypes_mapping.csv')

data_cluster = pd.merge(data, player_cluster, on='PLAYER_ID', how='left')
data_cluster.to_csv("data/shotlog_2014-15_with_clusters.csv")
print("Saved csv")


Saved csv


In [ ]:
cluster_zone_stat = data_cluster.groupby(['SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA','ARCHETYPE_CLUSTER'])['SHOT_MADE_FLAG'].agg(['count', 'sum'])
cluster_zone_stat['ACCURACY'] = cluster_zone_stat['sum']/cluster_zone_stat['count']

#cluster_zone_stat = cluster_zone_stat.reset_index()

print(cluster_zone_stat)

cluster_zone_stat.to_csv('data/cluster_zone_stat.csv')
print("Saved csv")

                                                    count  sum  ACCURACY
SHOT_ZONE_BASIC   SHOT_ZONE_AREA ARCHETYPE_CLUSTER                      
Above the Break 3 Back Court(BC) 0.0                    1    0  0.000000
                  Center(C)      0.0                 1258  452  0.359300
                                 1.0                 1221  420  0.343980
                                 2.0                  160   58  0.362500
                                 3.0                   11    6  0.545455
...                                                   ...  ...       ...
Right Corner 3    Right Side(R)  1.0                  645  255  0.395349
                                 2.0                  153   53  0.346405
                                 3.0                   22    3  0.136364
                                 4.0                 2203  905  0.410803
                                 5.0                  530  208  0.392453

[85 rows x 3 columns]
Saved csv


In [24]:
data[['SHOT_ZONE_RANGE','SHOT_ZONE_BASIC','SHOT_ZONE_AREA']].head()

,SHOT_ZONE_RANGE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA
0,Less Than 8 ft.,In The Paint (Non-RA),Center(C)
1,Less Than 8 ft.,Restricted Area,Center(C)
2,Less Than 8 ft.,Restricted Area,Center(C)
3,Less Than 8 ft.,In The Paint (Non-RA),Center(C)
4,24+ ft.,Above the Break 3,Right Side Center(RC)


In [ ]:
print(data['SHOT_ZONE_BASIC'].nunique())
data['SHOT_ZONE_BASIC'].value_counts()

7


SHOT_ZONE_BASIC
Mid-Range                29133
Above the Break 3        23915
Restricted Area          23705
In The Paint (Non-RA)    14630
Left Corner 3             4387
Right Corner 3            4176
Backcourt                   11
Name: count, dtype: int64

In [25]:
data.groupby(['SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA']).size()


SHOT_ZONE_BASIC        SHOT_ZONE_AREA       
Above the Break 3      Back Court(BC)               1
                       Center(C)                 5472
                       Left Side Center(LC)      8509
                       Right Side Center(RC)     8485
In The Paint (Non-RA)  Center(C)                12382
                       Left Side(L)               802
                       Right Side(R)             1015
Left Corner 3          Left Side(L)              4216
Mid-Range              Center(C)                 5571
                       Left Side Center(LC)      4344
                       Left Side(L)              6764
                       Right Side Center(RC)     5038
                       Right Side(R)             6471
Restricted Area        Center(C)                22821
Right Corner 3         Right Side(R)             4001
dtype: int64

In [8]:
data.groupby('SHOT_ZONE_BASIC')['SHOT_MADE_FLAG'].agg(['sum','count','mean'])

,sum,count,mean
SHOT_ZONE_BASIC,,,
Above the Break 3,7928,22467,0.352873
In The Paint (Non-RA),5768,14199,0.406226
Left Corner 3,1633,4216,0.387334
Mid-Range,11543,28188,0.409500
Restricted Area,14081,22821,0.617019
Right Corner 3,1596,4001,0.398900


In [ ]:
data.groupby(['SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA'])['SHOT_MADE_FLAG'].agg(['sum', 'count', 'mean'])

sum  count      mean
SHOT_ZONE_BASIC       SHOT_ZONE_AREA                               
Above the Break 3     Back Court(BC)             0      1  0.000000
                      Center(C)               1911   5472  0.349232
                      Left Side Center(LC)    2969   8509  0.348925
                      Right Side Center(RC)   3048   8485  0.359222
In The Paint (Non-RA) Center(C)               5003  12382  0.404054
                      Left Side(L)             340    802  0.423940
                      Right Side(R)            425   1015  0.418719
Left Corner 3         Left Side(L)            1633   4216  0.387334
Mid-Range             Center(C)               2296   5571  0.412134
                      Left Side Center(LC)    1766   4344  0.406538
                      Left Side(L)            2727   6764  0.403164
                      Right Side Center(RC)   2033   5038  0.403533
                      Right Side(R)           2721   6471  0.420491
Restricted Area       Center(C)              14081  22821  0.617019
Right Corner 3        Right Side(R)           1596   4001  0.398900

In [26]:
data_cluster.groupby(['SHOT_ZONE_BASIC','ARCHETYPE_CLUSTER'])['SHOT_MADE_FLAG'].agg(['sum','count','mean'])

sum  count      mean
SHOT_ZONE_BASIC       ARCHETYPE_CLUSTER                       
Above the Break 3     0.0                1575   4477  0.351798
                      1.0                1718   4943  0.347562
                      2.0                 195    554  0.351986
                      3.0                  14     43  0.325581
                      4.0                3571   9873  0.361694
                      5.0                 855   2577  0.331781
In The Paint (Non-RA) 0.0                1069   2647  0.403853
                      1.0                 747   1845  0.404878
                      2.0                1500   3611  0.415397
                      3.0                 773   1963  0.393785
                      4.0                 790   1979  0.399192
                      5.0                 888   2152  0.412639
Left Corner 3         0.0                 193    499  0.386774
                      1.0                 250    657  0.380518
                      2.0                  57    187  0.304813
                      3.0                  11     22  0.500000
                      4.0                 889   2283  0.389400
                      5.0                 233    568  0.410211
Mid-Range             0.0                2192   5418  0.404577
                      1.0                1893   4531  0.417789
                      2.0                2212   5412  0.408721
                      3.0                 351    973  0.360740
                      4.0                2104   5236  0.401833
                      5.0                2791   6618  0.421729
Restricted Area       0.0                2480   4258  0.582433
                      1.0                1521   2574  0.590909
                      2.0                3199   4948  0.646524
                      3.0                2864   4490  0.637862
                      4.0                1999   3353  0.596183
                      5.0                1992   3161  0.630180
Right Corner 3        0.0                 172    448  0.383929
                      1.0                 255    645  0.395349
                      2.0                  53    153  0.346405
                      3.0                   3     22  0.136364
                      4.0                 905   2203  0.410803
                      5.0                 208    530  0.392453